[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/15_mlp.ipynb)

# 🟠 Medium: SwiGLU MLP

Implement the **SwiGLU MLP** (feed-forward network) used in modern LLMs like LLaMA.

$$\text{SwiGLU}(x) = \text{down\_proj}\big(\text{SiLU}(\text{gate\_proj}(x)) \odot \text{up\_proj}(x)\big)$$

where $\text{SiLU}(x) = x \cdot \sigma(x)$

### Signature
```python
class SwiGLUMLP(nn.Module):
    def __init__(self, d_model: int, d_ff: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.gate_proj`: `nn.Linear(d_model, d_ff)`
- `self.up_proj`: `nn.Linear(d_model, d_ff)`
- `self.down_proj`: `nn.Linear(d_ff, d_model)`
- Activation: **SiLU** (a.k.a. Swish) — `F.silu` or implement as `x * torch.sigmoid(x)`

### Why SwiGLU?
Unlike the classic `Linear → ReLU/GELU → Linear` FFN, SwiGLU uses a **gating mechanism**:
the gate projection controls information flow, while the up projection provides the content.
This consistently outperforms standard FFNs in practice (PaLM, LLaMA, Mistral all use it).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.5 MB/s eta 0:00:00


In [2]:

import torch
import torch.nn as nn
import torch.nn.functional as F

In [9]:
# ✏️ YOUR IMPLEMENTATION HERE

class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff)
        self.up_proj = nn.Linear(d_model, d_ff)
        self.down_proj = nn.Linear(d_ff, d_model)
        # pass  # Initialize gate_proj, up_proj, down_proj

    def forward(self, x):
        y = self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))
        return y
        # pass  # down_proj(silu(gate_proj(x)) * up_proj(x))

In [10]:
# 🧪 Debug
mlp = SwiGLUMLP(d_model=64, d_ff=128)
x = torch.randn(2, 8, 64)
out = mlp(x)
print("Output shape:", out.shape)  # (2, 8, 64)
print("Params:", sum(p.numel() for p in mlp.parameters()))

Output shape: torch.Size([2, 8, 64])
Params: 24896


In [11]:
# ✅ SUBMIT
from torch_judge import check
check('mlp')


🧪 Testing: SwiGLU MLP (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Parameter shapes (1.9ms)
  ✅ [2/5] Forward output shape (1.0ms)
  ✅ [3/5] Numerical correctness (2.8ms)
  ✅ [4/5] 2-D input (0.7ms)
  ✅ [5/5] Gradient flow (1.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (8.1ms total)
  Progress saved. Run status() to see your dashboard.

